# F4 — Análisis de resultados SIMCE 2025
**4° Básico · Matemática**

Este cuaderno implementa el análisis final acordado para responder las tres preguntas de investigación:

1. ¿Existen diferencias estadísticamente significativas en los puntajes promedio según ubicación geográfica?
2. ¿Qué regiones, provincias y comunas concentran establecimientos con desempeño inferior a la referencia nacional?
3. ¿Qué establecimientos y territorios pueden identificarse como potenciales focos de priorización?

### Decisiones metodológicas
- La **unidad de análisis inferencial** es el establecimiento.
- Para comparar establecimientos territorialmente se usa el **promedio simple**.
- Para estimar desempeño territorial y construir la referencia de criticidad se usa el **promedio ponderado por estudiantes**.
- El ICT comunal combina **déficit (50%) + concentración (30%) + exposición (20%)**, normalizados mediante percentiles.
- Se selecciona el **P90 del ICT** como criterio de focalización relativa.
- La sensibilidad se comprueba excluyendo comunas bajo P10 del número de establecimientos.

## 1. Configuración, carga y validaciones

In [2]:
from pathlib import Path
from importlib import metadata
import sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.oneway import anova_oneway, effectsize_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

candidatos = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidatos if (p/'src/poo.py').is_file() and (p/'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro del repositorio SIMCE con src/ y data/.')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.analisis_f4 import (validar_entrada, promedio_ponderado, resumen_criticidad,
    calcular_ict, prueba_territorial, REGION, COMUNA)

SEMILLA = 42
CUANTIL_CRITICO = 0.90
np.random.seed(SEMILLA)
OUT = ROOT/'F4/resultados'
OUT.mkdir(parents=True, exist_ok=True)
paquetes = ['numpy','pandas','matplotlib','nbformat','nbclient','ipykernel','ipython']
versiones = {p: metadata.version(p) for p in paquetes}
print('Python:', platform.python_version(), '| Plataforma:', platform.platform())
print('Entorno virtual:', sys.prefix != sys.base_prefix)
display(pd.DataFrame(versiones.items(), columns=['paquete','version']))
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

ModuleNotFoundError: No module named 'scipy'

## III. Continuidad y decisiones F2/F3

| Decisión mantenida | Alternativa descartada y razón específica |
|---|---|
| Efectiva: alumnos > 0 y sin observación de puntaje | Reincorporar marca 2 por tener puntaje contradice la advertencia de representatividad |
| No imputar RBD, geografía ni puntaje | Un valor inventado alteraría pertenencia territorial o prioridad de intervención |
| Conservar extremos válidos | Eliminar escuelas grandes o con puntaje extremo borraría precisamente casos relevantes |
| Mantener puntaje y alumnos en sus unidades | Un z-score no responde cuántos alumnos se asocian a cuántos puntos de brecha |
| Geografía embebida y unión validada | Una tabla externa adicional puede desalinear códigos entre ejecuciones |
| Errores detienen el proceso | Eliminar filas inválidas silenciosamente cambiaría el universo y los denominadores |

**Arquitectura y POO.** `Transformador` es una clase abstracta; cinco hijas implementan `transformar`. `PipelineSIMCE` compone esas instancias y las recorre por su interfaz común (polimorfismo). Las propiedades exponen copias o tuplas, evitando que el consumidor altere accidentalmente el registro interno. Se reutiliza esta arquitectura, sin crear otra jerarquía artificial en F4.

Separar `auditoria.py` de `validacion.py` permite conservar evidencia de exclusiones mientras los errores de integridad detienen la ejecución. `configuracion.py` centraliza catálogos para que las clases y las funciones compartan reglas. `poo.py` coordina las transformaciones sin duplicar su lógica. `analisis_f4.py` recibe el producto ya validado: cambiar un gráfico no debe modificar la regla de efectividad. Estas decisiones atienden las observaciones formativas del 25 de septiembre.

In [ ]:
from src.configuracion import COLUMNAS_SIMCE
from src.carga import cargar_simce, construir_dim_geografia
from src.poo import (PipelineSIMCE, TransformadorTipos, TransformadorTextos,
    TransformadorCategorias, TransformadorGeografia, TransformadorEfectividad)
from src.auditoria import construir_auditoria
from src.transformacion import construir_cobertura_regional
from src.validacion import validar_dataset_final
from src.algoritmo import construir_jerarquia_geografica, validar_geografia_recursiva

raw_path = ROOT/'data/raw/simce4b2025_rbd_final.csv'
f2_path = ROOT/'data/processed/simce4b2025_matematica_efectiva.csv'
df_f2 = pd.read_csv(f2_path, encoding='utf-8-sig', dtype={'no_aplica':'string'}, parse_dates=['fecha_bbdd'])
dim = construir_dim_geografia()
bruto = cargar_simce(raw_path, COLUMNAS_SIMCE)
pipeline = PipelineSIMCE([TransformadorTipos(), TransformadorTextos(),
    TransformadorCategorias(), TransformadorGeografia(dim), TransformadorEfectividad()])
transformado = pipeline.ejecutar(bruto)
reconstruido = pipeline.construir_producto_final(transformado)
auditoria, resumen_auditoria, resumen_marcas = construir_auditoria(transformado)
cobertura = construir_cobertura_regional(transformado)
controles_f3 = validar_dataset_final(reconstruido, transformado, auditoria, cobertura)
cols = sorted(reconstruido.columns)
pd.testing.assert_frame_equal(
    df_f2.sort_values('rbd').reset_index(drop=True)[cols],
    reconstruido.sort_values('rbd').reset_index(drop=True)[cols], check_dtype=False)
assert len(bruto) == len(df_f2) + len(auditoria)
jerarquia = construir_jerarquia_geografica(dim)
assert validar_geografia_recursiva(jerarquia) == []
df = validar_entrada(df_f2)
print(f'Bruto: {len(bruto):,}; efectivos: {len(df):,}; auditados: {len(auditoria):,}.')
print('Igualdad de los 30 campos F2/F3 comprobada; controles:', len(controles_f3))
display(pd.DataFrame(pipeline.registro))
display(cobertura)
comunas_ausentes = dim.loc[~dim.cod_com_rbd.isin(df.cod_com_rbd), ['cod_reg_rbd','region','cod_com_rbd','comuna']]
print('Comunas de la dimensión sin establecimientos efectivos:')
display(comunas_ausentes)

In [ ]:
# Normalización de nombres mínimos usados en F4
ren = {
    "Puntaje promedio": "puntaje_promedio",
    "Exposición": "exposicion",
    "Bajo desempeño": "bajo_desempeno",
}
df = df.rename(columns={k:v for k,v in ren.items() if k in df.columns})

requeridas = [
    "rbd", "region", "provincia", "cod_com_rbd", "comuna",
    "n_alumnos", "puntaje_promedio", "grupo_socioeconomico",
    "ruralidad", "dependencia_4_cat"
]
faltantes = [c for c in requeridas if c not in df.columns]
assert not faltantes, f"Faltan columnas requeridas: {faltantes}"

df["puntaje_promedio"] = pd.to_numeric(df["puntaje_promedio"], errors="coerce")
df["n_alumnos"] = pd.to_numeric(df["n_alumnos"], errors="coerce")

assert df["rbd"].notna().all(), "Hay RBD vacíos."
assert not df["rbd"].duplicated().any(), "Hay RBD duplicados."
assert df["puntaje_promedio"].notna().all(), "Hay puntajes faltantes."
assert df["n_alumnos"].notna().all(), "Hay n_alumnos faltantes."
assert (df["n_alumnos"] > 0).all(), "Hay establecimientos sin estudiantes válidos."

print("✓ Validaciones mínimas superadas.")

## 2. Caracterización general y referencias nacionales

In [ ]:
n_ee = df["rbd"].nunique()
n_est = int(df["n_alumnos"].sum())
n_reg = df["region"].nunique()
n_prov = df["provincia"].nunique()
n_com = df["cod_com_rbd"].nunique()

promedio_simple_nacional = df["puntaje_promedio"].mean()
promedio_ponderado_nacional = np.average(df["puntaje_promedio"], weights=df["n_alumnos"])

resumen_general = pd.DataFrame({
    "Indicador": [
        "Establecimientos", "Estudiantes", "Regiones", "Provincias", "Comunas",
        "Promedio simple EE", "Promedio nacional ponderado", "Mediana EE", "DE EE"
    ],
    "Valor": [
        n_ee, n_est, n_reg, n_prov, n_com,
        promedio_simple_nacional, promedio_ponderado_nacional,
        df["puntaje_promedio"].median(), df["puntaje_promedio"].std(ddof=1)
    ]
})

# Crear una versión solo para visualización
resumen_mostrar = resumen_general.copy()

indicadores_enteros = [
    "Establecimientos",
    "Estudiantes",
    "Regiones",
    "Provincias",
    "Comunas"
]

resumen_mostrar["Valor"] = resumen_mostrar.apply(
    lambda x: (
        f"{x['Valor']:.0f}"
        if x["Indicador"] in indicadores_enteros
        else f"{x['Valor']:.2f}"
    ),
    axis=1
)

display(resumen_mostrar)
print(f"Referencia ponderada usada para criticidad: {promedio_ponderado_nacional:.2f}")

## 3. Pregunta 1 — ¿Existen diferencias territoriales en los puntajes?

Antes de identificar territorios críticos, necesitamos responder una pregunta previa: **¿la ubicación geográfica está asociada con diferencias en los puntajes de los establecimientos?**

Para mantener el análisis simple, se realiza una prueba global en tres escalas territoriales: **región, provincia y comuna**. El objetivo de esta etapa **no es seleccionar regiones ni construir un ranking**, sino comprobar si existen diferencias entre territorios que justifiquen avanzar hacia un análisis más detallado.

### ¿Cómo se interpreta?
- La hipótesis nula plantea que los territorios tienen, en promedio, el mismo desempeño.
- Si `p < 0,05`, existe evidencia estadística de que **al menos algunos territorios difieren entre sí**.
- Como las dispersiones no son homogéneas, el cuaderno utiliza **ANOVA de Welch**, una versión más robusta ante diferencias de varianza y tamaños de grupo.
- Junto al valor p se presenta un **tamaño de efecto (η² equivalente)**, que ayuda a distinguir entre una diferencia estadísticamente detectable y una diferencia territorial de mayor magnitud.

Esta prueba funciona como un **puente metodológico**: primero verificamos que las diferencias territoriales existen y luego pasamos a la pregunta sustantiva del proyecto: **¿dónde se concentra el bajo desempeño y qué territorios podrían priorizarse?**

In [ ]:
def descriptivos_territoriales(datos, geo):
    return (
        datos.groupby(geo)["puntaje_promedio"]
        .agg(N="count", Media="mean", Mediana="median", DE="std",
             P25=lambda x: x.quantile(.25), P75=lambda x: x.quantile(.75))
        .reset_index()
        .sort_values("Media")
    )

desc_region = descriptivos_territoriales(df, "region")
desc_provincia = descriptivos_territoriales(df, "provincia")
desc_comuna = descriptivos_territoriales(df, "comuna")

display(desc_region.round(2))

### Gráfico 1 — Distribución de puntajes por región

Este gráfico acompaña la prueba estadística y permite observar que los puntajes **no se distribuyen exactamente de la misma manera en todas las regiones**. Sin embargo, la región se utiliza aquí solo como una visualización sintética de las diferencias territoriales.

**Importante:** el análisis no continúa seleccionando una región específica. La focalización se realizará a nivel **comunal**, porque esa escala permite identificar territorios más concretos para una eventual priorización y evita que los promedios regionales oculten diferencias internas.

In [ ]:
orden_regiones = (
    df[["region", "Orden"]]
    .drop_duplicates()
    .sort_values("Orden")["region"]
    .tolist()
)
datos_box = [df.loc[df["region"].eq(r), "puntaje_promedio"].to_numpy() for r in orden_regiones]

fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(datos_box, tick_labels=orden_regiones, orientation="horizontal", showfliers=False)
ax.axvline(promedio_simple_nacional, linestyle="--",
           label=f"Promedio simple nacional ({promedio_simple_nacional:.1f})")
ax.set_title("Distribución de puntajes SIMCE Matemática 4° Básico por región")
ax.set_xlabel("Puntaje promedio del establecimiento")
ax.set_ylabel("Región")
ax.legend()
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

### Principales hallazgos descriptivos del Gráfico 1

- Las distribuciones de puntajes **se superponen ampliamente entre regiones**, por lo que no se observa una separación absoluta entre territorios: dentro de todas las regiones existe heterogeneidad entre establecimientos.
- Aun con ese solapamiento, las **medianas y los rangos intercuartílicos no son idénticos**, lo que hace creer en la existencia de diferencias territoriales que posteriormente serán analizadas mediante pruebas estadísticas.
- La línea del promedio simple nacional sirve como referencia para observar que algunas distribuciones regionales se concentran relativamente más hacia puntajes bajos y otras hacia puntajes altos.
- El gráfico debe interpretarse como una **visión general del componente territorial**, no como un ranking de regiones. Su principal aporte es mostrar que el desempeño varía tanto **entre regiones como dentro de ellas**.

**Conexión con el análisis siguiente.** La comparación regional permite visualizar la existencia de heterogeneidad territorial, pero una región es una unidad demasiado amplia para la focalización. Por ello, después de comprobar estadísticamente las diferencias, el análisis avanza hacia la **escala comunal**, que permite localizar con mayor precisión dónde se concentra el bajo desempeño.

In [ ]:
def prueba_territorial(datos, geo):
    grupos = [
        g["puntaje_promedio"].dropna().astype(float)
        for _, g in datos.groupby(geo)
        if g["puntaje_promedio"].notna().sum() >= 2
    ]

    # Se verifica homogeneidad de varianzas.
    _, p_levene = stats.levene(*grupos, center="median")

    # Si las varianzas son diferentes, se utiliza Welch.
    tipo = "equal" if p_levene >= 0.05 else "unequal"
    nombre_prueba = "ANOVA clásico" if tipo == "equal" else "ANOVA de Welch"
    resultado = anova_oneway(grupos, use_var=tipo, welch_correction=True)

    # Tamaño de efecto global.
    medias = np.array([g.mean() for g in grupos])
    varianzas = np.array([g.var(ddof=1) for g in grupos])
    ns = np.array([len(g) for g in grupos])
    f2 = float(effectsize_oneway(medias, varianzas, ns, use_var=tipo))
    eta2 = f2 / (1 + f2)

    return {
        "Escala territorial": geo.capitalize(),
        "N territorios": len(grupos),
        "Prueba": nombre_prueba,
        "p": float(resultado.pvalue),
        "η² equivalente": eta2
    }

resultados_pruebas = pd.DataFrame([
    prueba_territorial(df, "region"),
    prueba_territorial(df, "provincia"),
    prueba_territorial(df, "comuna"),
])

resultados_pruebas["Conclusión"] = np.where(
    resultados_pruebas["p"] < 0.05,
    "Se observan diferencias territoriales",
    "No se detectan diferencias territoriales"
)

display(resultados_pruebas.round({"p": 6, "η² equivalente": 3}))

### Lectura de este resultado

Los resultados permiten responder la primera pregunta de investigación: **sí se observan diferencias estadísticamente significativas en los puntajes de los establecimientos según su ubicación geográfica**.

La magnitud de esas diferencias aumenta al pasar desde escalas territoriales amplias hacia escalas más específicas. En este conjunto de datos, el tamaño de efecto es aproximadamente **0,032 a nivel regional, 0,048 a nivel provincial y 0,241 a nivel comunal**. Esto sugiere que una parte importante de la heterogeneidad territorial aparece cuando se observa el territorio con mayor detalle.

Este resultado orienta el resto del análisis: **no necesitamos identificar cuál región “gana” o “pierde” en el ANOVA**. Lo relevante es que existe heterogeneidad territorial y que la escala comunal ofrece una mayor capacidad para distinguirla. Por ello, desde este punto el análisis se concentra en las comunas y en la identificación de aquellas que combinan **bajo desempeño, concentración de establecimientos bajo la referencia y cantidad de estudiantes expuestos**.